# M3L2 E00 - Chat basico y PromptTemplate

## Objetivo

Vas a conectar con un modelo de chat usando LangChain y vas a ver por que `PromptTemplate` ayuda a dejar los prompts ordenados.

```text
Pregunta del usuario -> Prompt -> Modelo -> Respuesta
```

## Entregable

- una primera llamada al modelo con `ChatOpenAI`;
- un `PromptTemplate` con una variable;
- una respuesta generada desde el template;
- pequenos cambios de practica sobre el mismo patron.

## Conceptos base

| Concepto | Definicion simple | En codigo |
|---|---|---|
| LLM | Modelo que genera texto a partir de instrucciones | `llm.invoke(...)` |
| Prompt | Instruccion que recibe el modelo | string o template |
| Template | Prompt reutilizable con variables | `{concepto}` |
| API key | Credencial para usar OpenAI | `getpass` en el notebook |

No hardcodeamos la API key porque el notebook se puede compartir. En Colab la pedimos con `getpass` y la guardamos solo en la variable de entorno de la sesion.

## Diagrama del flujo

```text
getpass -> OPENAI_API_KEY -> ChatOpenAI
                              |
                              v
                       PromptTemplate
                              |
                              v
                         modelo.invoke()
                              |
                              v
                           AIMessage
```

In [ ]:
# !pip install langchain langchain-openai

In [ ]:
import os
import getpass
from langchain_openai import ChatOpenAI

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Ingresa tu OpenAI API key: ")


def obtener_modelo(temperature: float = 0.2):
    return ChatOpenAI(model="gpt-4o-mini", temperature=temperature)


print("API key cargada en la variable de entorno OPENAI_API_KEY")

In [ ]:
llm = obtener_modelo()
print(type(llm))

## Paso 1 - Primer invoke manual

Antes de usar templates, probamos la llamada mas simple: enviar un string directo al modelo.

In [ ]:
respuesta = llm.invoke("Explica en una frase que es LangChain para un alumno principiante.")
print(respuesta.content)

## Paso 2 - PromptTemplate

Un string manual funciona para algo chico. Cuando el prompt tiene variables, rol, formato y restricciones, conviene convertirlo en un objeto.

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    input_variables=["concepto"],
    template=(
        "Explica el concepto '{concepto}' para un alumno principiante. "
        "Usa una analogia de cocina y responde en maximo 5 lineas."
    ),
)

print(prompt)
print("Variables:", prompt.input_variables)

In [ ]:
prompt_renderizado = prompt.format(concepto="PromptTemplate")
print(prompt_renderizado)

In [ ]:
respuesta = llm.invoke(prompt_renderizado)
print(respuesta.content)

In [ ]:
for concepto in ["LCEL", "ChatOpenAI", "OutputParser"]:
    texto = prompt.format(concepto=concepto)
    print("\n---", concepto, "---")
    print(llm.invoke(texto).content)

## Practica

1. Cambia el concepto a `LCEL`.
2. Cambia la analogia de cocina por una de transporte.
3. Crea otro template para conceptos de programacion.
4. Inspecciona `prompt.input_variables`.
5. Explica por que este ejemplo todavia no tiene memoria, RAG ni tools.

In [ ]:
assert prompt is not None
assert "concepto" in prompt.input_variables
assert isinstance(prompt_renderizado, str)
assert respuesta is not None
print("Checks OK")

## Resumen

`PromptTemplate -> ChatOpenAI -> AIMessage` es el bloque mas basico de LangChain.